In [ ]:
import pandas as pd
import MySQLdb
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
import json
import requests
import urllib

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py


db = client.get_database('telematics_db')



In [ ]:
no_info=pd.DataFrame(list(db.deviceHistory.find({'fromTime':{'$gte':(tm.time()-86400*4)},'toTime':{'$lte':tm.time()}})))

In [ ]:
vehicle_details=pd.read_sql("select vehicle_id,network,sim_phone_number,model_name from vehicle_details where vehicle_id in {} ".format(tuple(no_info['vehicleId'].unique())),con=engine)

In [ ]:
data=pd.merge(no_info,vehicle_details,how='left',left_on='vehicleId',right_on='vehicle_id')

In [ ]:
data=data[data['network']=='VODAFONE']

In [ ]:
device_history=data

In [ ]:
pings=pd.DataFrame()
sortOrder="desc"
timeCondition="lte"
for i, row in device_history.iterrows():
        from_tm=int(row['fromTime'])
        to_tm=""
        
        
        url=GPS_API_BASE+"/gps/top?vehicleIds={}&fromTime={}&toTime={}&sortOrder={}&timeCondition={}".format(row['vehicleId'],from_tm,to_tm,sortOrder,timeCondition)
        data=json.load(urllib.request.urlopen(url))
        
        try:
            #print(data)
            data1=pd.DataFrame([[data['data'][str(row['vehicleId'])]['location'][0],data['data'][str(row['vehicleId'])]['location'][1],data['data'][str(row['vehicleId'])]['vehicleId']]],columns=['longitude','latitude','vehicleId'])
            pings=pd.concat([pings,data1])
            
        except:
            
            print('no_DATA')
            

In [ ]:
len(pings)

In [ ]:
import folium
from folium import plugins
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
m = folium.Map([27, 78], zoom_start=5)


for index, row in pings.iterrows():
    folium.CircleMarker([row['latitude'], row['longitude']],
                        radius=3,
                        popup=str(str(row['latitude'])+str(' ,' + str(row['longitude']) )),
                        #fill_color="#3db7e4", # divvy color
                       ).add_to(m)
m.save('no_info_points.html')